# Advanced Model 2 Hard Negative: Blend + Fused Embedding Re-rank

This version keeps the two-stage structure, but changes stage 2 from a scalar cross-attention scorer into a bidirectional fused embedding reranker.

Main idea:
- stage 1: dual-encoder coarse retrieval
- stage 2: bidirectional cross-attention fused embedding for reranking
- final score: blend coarse similarity and fused similarity


In [ ]:
import os
from dataclasses import dataclass
from typing import Dict, List, Optional, Sequence

import torch
import torch.nn as nn
import torch.nn.functional as F
from datasets import load_dataset
from torch.utils.data import DataLoader
from tqdm import tqdm
from transformers import BertModel, BertTokenizer, ViTImageProcessor, ViTModel


In [ ]:
@dataclass
class Config:
    dataset_name: str = "Zoe3324/flickr30k-pairs"
    text_model_name: str = "bert-base-uncased"
    image_model_name: str = "google/vit-base-patch16-224"
    max_text_len: int = 32
    embed_dim: int = 256
    batch_size: int = 32
    epochs: int = 1
    lr: float = 1e-5
    temperature: float = 0.07
    hard_negative_top_k: int = 3
    fused_loss_weight: float = 0.2
    eval_num_samples: int = 1000
    eval_top_k: int = 20
    blend_alpha: float = 0.5
    num_workers: int = 2
    checkpoint_path: str = "/content/advanced_model_2_hard_negative_blend_fused_embedding.pt"


cfg = Config()
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(device)


cuda


In [ ]:
dataset = load_dataset(cfg.dataset_name)
train_data = dataset["train"]
val_data = dataset["validation"]
test_data = dataset["test"]

print(train_data[0])


README.md:   0%|          | 0.00/657 [00:00<?, ?B/s]

data/train-00000-of-00012.parquet:   0%|          | 0.00/97.0M [00:00<?, ?B/s]

data/train-00001-of-00012.parquet:   0%|          | 0.00/96.2M [00:00<?, ?B/s]

data/train-00002-of-00012.parquet:   0%|          | 0.00/101M [00:00<?, ?B/s]

data/train-00003-of-00012.parquet:   0%|          | 0.00/99.0M [00:00<?, ?B/s]

...

data/test-00000-of-00001.parquet:   0%|          | 0.00/40.1M [00:00<?, ?B/s]

Generating train split:   0%|          | 0/145000 [00:00<?, ? examples/s]

Generating validation split:   0%|          | 0/5070 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/5000 [00:00<?, ? examples/s]

{'image': <PIL.JpegImagePlugin.JpegImageFile image mode=RGB size=333x500 at 0x7C94CF072E10>, 'caption': 'Two young guys with shaggy hair look at their hands while hanging out in the yard.', 'split': 'train', 'img_id': '0', 'filename': '1000092795.jpg'}


In [ ]:
tokenizer = BertTokenizer.from_pretrained(cfg.text_model_name)
image_processor = ViTImageProcessor.from_pretrained(cfg.image_model_name)


tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

preprocessor_config.json:   0%|          | 0.00/160 [00:00<?, ?B/s]

In [ ]:
class CrossAttention(nn.Module):
    def __init__(self, dim, num_heads=8, dropout=0.1, ffn_mult=4):
        super().__init__()
        self.q_norm = nn.LayerNorm(dim)
        self.k_norm = nn.LayerNorm(dim)
        self.v_norm = nn.LayerNorm(dim)
        self.ffn_norm = nn.LayerNorm(dim)
        self.attn = nn.MultiheadAttention(dim, num_heads=num_heads, dropout=dropout, batch_first=True)
        self.ffn = nn.Sequential(
            nn.Linear(dim, dim * ffn_mult),
            nn.GELU(),
            nn.Dropout(dropout),
            nn.Linear(dim * ffn_mult, dim),
            nn.Dropout(dropout),
        )

    def forward(self, q, k, v):
        q_in = self.q_norm(q)
        k_in = self.k_norm(k)
        v_in = self.v_norm(v)
        attn_out, _ = self.attn(q_in, k_in, v_in, need_weights=False)
        x = q + attn_out
        x = x + self.ffn(self.ffn_norm(x))
        return x


class TwoStageFusedEmbeddingModel(nn.Module):
    def __init__(self, embed_dim=256, temperature=0.07):
        super().__init__()
        self.text_encoder = BertModel.from_pretrained(cfg.text_model_name)
        self.image_encoder = ViTModel.from_pretrained(cfg.image_model_name)

        dim = self.text_encoder.config.hidden_size
        self.image_proj = nn.Linear(dim, embed_dim)
        self.text_proj = nn.Linear(dim, embed_dim)

        self.text_to_image_attn = CrossAttention(dim)
        self.image_to_text_attn = CrossAttention(dim)
        self.fused_proj = nn.Linear(dim, embed_dim)

        self.temperature = nn.Parameter(torch.tensor(temperature, dtype=torch.float32))

    @staticmethod
    def masked_mean_pool(hidden_states, attention_mask):
        mask = attention_mask.unsqueeze(-1).float()
        pooled = (hidden_states * mask).sum(dim=1)
        denom = mask.sum(dim=1).clamp(min=1.0)
        return pooled / denom

    def encode_image(self, pixel_values):
        image_tokens = self.image_encoder(pixel_values=pixel_values).last_hidden_state
        image_emb = F.normalize(self.image_proj(image_tokens[:, 0]), dim=-1)
        return image_emb, image_tokens

    def encode_text(self, input_ids, attention_mask):
        text_tokens = self.text_encoder(input_ids=input_ids, attention_mask=attention_mask).last_hidden_state
        text_pooled = self.masked_mean_pool(text_tokens, attention_mask)
        text_emb = F.normalize(self.text_proj(text_pooled), dim=-1)
        return text_emb, text_tokens

    def compute_fused_embedding(self, text_tokens, image_tokens, attention_mask):
        text_fused = self.text_to_image_attn(text_tokens, image_tokens, image_tokens)
        image_fused = self.image_to_text_attn(image_tokens, text_tokens, text_tokens)
        text_pooled = self.masked_mean_pool(text_fused, attention_mask)
        image_pooled = image_fused[:, 0]
        fused = 0.5 * (text_pooled + image_pooled)
        fused_emb = F.normalize(self.fused_proj(fused), dim=-1)
        return fused_emb

    def similarity(self, left, right):
        return left @ right.T / self.temperature.clamp(min=1e-4)

    def forward(self, input_ids, attention_mask, pixel_values):
        image_emb, image_tokens = self.encode_image(pixel_values)
        text_emb, text_tokens = self.encode_text(input_ids, attention_mask)
        sim = self.similarity(image_emb, text_emb)
        fused_emb = self.compute_fused_embedding(text_tokens, image_tokens, attention_mask)
        fused_sim = self.similarity(fused_emb, fused_emb)
        return {
            "image_emb": image_emb,
            "text_emb": text_emb,
            "image_tokens": image_tokens,
            "text_tokens": text_tokens,
            "fused_emb": fused_emb,
            "sim": sim,
            "fused_sim": fused_sim,
        }


model = TwoStageFusedEmbeddingModel(embed_dim=cfg.embed_dim, temperature=cfg.temperature).to(device)


config.json:   0%|          | 0.00/570 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/440M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BertModel LOAD REPORT from: bert-base-uncased
Key                                        | Status     |  | 
-------------------------------------------+------------+--+-
cls.predictions.transform.dense.weight     | UNEXPECTED |  | 
cls.predictions.transform.dense.bias       | UNEXPECTED |  | 
cls.predictions.bias                       | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED |  | 
cls.seq_relationship.weight                | UNEXPECTED |  | 
cls.seq_relationship.bias                  | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


config.json: 0.00B [00:00, ?B/s]

model.safetensors:   0%|          | 0.00/346M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/198 [00:00<?, ?it/s]

ViTModel LOAD REPORT from: google/vit-base-patch16-224
Key                 | Status     | 
--------------------+------------+-
classifier.bias     | UNEXPECTED | 
classifier.weight   | UNEXPECTED | 
pooler.dense.weight | MISSING    | 
pooler.dense.bias   | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


In [ ]:
def collate_fn(batch: Sequence[Dict]) -> Dict[str, torch.Tensor]:
    texts = [x["caption"] for x in batch]
    images = [x["image"].convert("RGB") for x in batch]

    text_enc = tokenizer(
        texts,
        padding="max_length",
        truncation=True,
        max_length=cfg.max_text_len,
        return_tensors="pt",
    )
    image_enc = image_processor(images=images, return_tensors="pt")

    return {
        "input_ids": text_enc["input_ids"],
        "attention_mask": text_enc["attention_mask"],
        "pixel_values": image_enc["pixel_values"],
    }


In [ ]:
def contrastive_loss(sim):
    labels = torch.arange(sim.size(0), device=sim.device)
    loss_i2t = F.cross_entropy(sim, labels)
    loss_t2i = F.cross_entropy(sim.T, labels)
    return 0.5 * (loss_i2t + loss_t2i)


def stage2_topk_rerank_loss(model, outputs, attention_mask):
    coarse_scores = outputs["image_emb"] @ outputs["text_emb"].T
    batch_size = coarse_scores.size(0)
    if batch_size < 2:
        return torch.tensor(0.0, device=coarse_scores.device)

    k = min(cfg.hard_negative_top_k + 1, batch_size)
    loss_i2t = 0.0
    loss_t2i = 0.0

    for i in range(batch_size):
        image_query = outputs["image_emb"][i:i+1]
        text_query = outputs["text_emb"][i:i+1]

        i2t_candidates = coarse_scores[i].topk(k).indices.tolist()
        if i not in i2t_candidates:
            i2t_candidates[-1] = i
        pos_i2t = i2t_candidates.index(i)

        cand_text_tokens = outputs["text_tokens"][i2t_candidates]
        cand_text_mask = attention_mask[i2t_candidates]
        query_image_tokens = outputs["image_tokens"][i:i+1].expand(len(i2t_candidates), -1, -1)
        cand_fused_i2t = model.compute_fused_embedding(cand_text_tokens, query_image_tokens, cand_text_mask)
        rerank_logits_i2t = (image_query @ cand_fused_i2t.T).squeeze(0) / model.temperature.clamp(min=1e-4)
        loss_i2t = loss_i2t + F.cross_entropy(rerank_logits_i2t.unsqueeze(0), torch.tensor([pos_i2t], device=coarse_scores.device))

        t2i_candidates = coarse_scores[:, i].topk(k).indices.tolist()
        if i not in t2i_candidates:
            t2i_candidates[-1] = i
        pos_t2i = t2i_candidates.index(i)

        cand_image_tokens = outputs["image_tokens"][t2i_candidates]
        query_text_tokens = outputs["text_tokens"][i:i+1].expand(len(t2i_candidates), -1, -1)
        query_text_mask = attention_mask[i:i+1].expand(len(t2i_candidates), -1)
        cand_fused_t2i = model.compute_fused_embedding(query_text_tokens, cand_image_tokens, query_text_mask)
        rerank_logits_t2i = (text_query @ cand_fused_t2i.T).squeeze(0) / model.temperature.clamp(min=1e-4)
        loss_t2i = loss_t2i + F.cross_entropy(rerank_logits_t2i.unsqueeze(0), torch.tensor([pos_t2i], device=coarse_scores.device))

    loss_i2t = loss_i2t / batch_size
    loss_t2i = loss_t2i / batch_size
    return 0.5 * (loss_i2t + loss_t2i)


def run_epoch(model, loader, optimizer=None):
    training = optimizer is not None
    model.train(training)
    total_loss = 0.0
    total_ctr = 0.0
    total_fused = 0.0

    for batch in tqdm(loader, desc="Train" if training else "Val"):
        input_ids = batch["input_ids"].to(device)
        attention_mask = batch["attention_mask"].to(device)
        pixel_values = batch["pixel_values"].to(device)

        outputs = model(input_ids=input_ids, attention_mask=attention_mask, pixel_values=pixel_values)
        loss_ctr = contrastive_loss(outputs["sim"])
        loss_fused = stage2_topk_rerank_loss(model, outputs, attention_mask)
        loss = loss_ctr + cfg.fused_loss_weight * loss_fused

        if training:
            optimizer.zero_grad()
            loss.backward()
            optimizer.step()

        total_loss += loss.item()
        total_ctr += loss_ctr.item()
        total_fused += loss_fused.item()

    num_batches = max(len(loader), 1)
    return {
        "loss": total_loss / num_batches,
        "contrastive": total_ctr / num_batches,
        "fused": total_fused / num_batches,
    }


train_loader = DataLoader(train_data, batch_size=cfg.batch_size, shuffle=True, collate_fn=collate_fn, num_workers=cfg.num_workers)
val_loader = DataLoader(val_data, batch_size=cfg.batch_size, shuffle=False, collate_fn=collate_fn, num_workers=cfg.num_workers)

optimizer = torch.optim.AdamW(model.parameters(), lr=cfg.lr)
best_val_loss = float("inf")

for epoch in range(cfg.epochs):
    print(f"\nEpoch {epoch + 1}/{cfg.epochs}")
    train_stats = run_epoch(model, train_loader, optimizer=optimizer)
    with torch.no_grad():
        val_stats = run_epoch(model, val_loader, optimizer=None)

    print(f"train_loss={train_stats['loss']:.4f} train_ctr={train_stats['contrastive']:.4f} train_fused={train_stats['fused']:.4f}")
    print(f"val_loss={val_stats['loss']:.4f} val_ctr={val_stats['contrastive']:.4f} val_fused={val_stats['fused']:.4f} temperature={model.temperature.detach().item():.4f}")

    if val_stats['loss'] < best_val_loss:
        best_val_loss = val_stats['loss']
        torch.save(model.state_dict(), cfg.checkpoint_path)
        print(f"saved best checkpoint to {cfg.checkpoint_path}")

if os.path.exists(cfg.checkpoint_path):
    model.load_state_dict(torch.load(cfg.checkpoint_path, map_location=device))
    print(f"reloaded best checkpoint from {cfg.checkpoint_path}")



Epoch 1/1


Val: 100%|██████████| 159/159 [00:41<00:00,  3.86it/s]


train_loss=0.6784 train_ctr=0.5660 train_fused=0.5623
val_loss=2.3186 val_ctr=1.9768 val_fused=1.7093 temperature=0.0413
saved best checkpoint to /content/advanced_model_2_hard_negative_blend_fused_embedding.pt
reloaded best checkpoint from /content/advanced_model_2_hard_negative_blend_fused_embedding.pt


In [ ]:
def eval_collate_fn(batch):
    texts = [x["caption"] for x in batch]
    images = [x["image"].convert("RGB") for x in batch]
    img_ids = [x["img_id"] for x in batch]
    return texts, images, img_ids


def recall_at_k(score_mat, k, img_ids=None):
    if img_ids is None:
        correct = 0
        for i in range(score_mat.size(0)):
            topk = score_mat[i].topk(k).indices.tolist()
            if i in topk:
                correct += 1
        return correct / score_mat.size(0)

    group_to_indices = {}
    for idx, gid in enumerate(img_ids):
        group_to_indices.setdefault(gid, []).append(idx)

    total_recall = 0.0
    for i in range(score_mat.size(0)):
        topk = set(score_mat[i].topk(k).indices.tolist())
        positives = set(group_to_indices[img_ids[i]])
        total_recall += len(topk & positives) / len(positives)
    return total_recall / score_mat.size(0)


def collect_embeddings(data, num_samples=None):
    if num_samples is not None:
        data = data.select(range(min(num_samples, len(data))))

    loader = DataLoader(data, batch_size=64, shuffle=False, collate_fn=eval_collate_fn)
    all_texts = []
    all_images = []
    all_img_ids = []
    all_image_embs = []
    all_text_embs = []

    model.eval()
    with torch.no_grad():
        for texts, images, img_ids in tqdm(loader, desc="Encoding"):
            text_enc = tokenizer(texts, padding="max_length", truncation=True, max_length=cfg.max_text_len, return_tensors="pt")
            image_enc = image_processor(images=images, return_tensors="pt")

            image_emb, _ = model.encode_image(image_enc["pixel_values"].to(device))
            text_emb, _ = model.encode_text(text_enc["input_ids"].to(device), text_enc["attention_mask"].to(device))

            all_image_embs.append(image_emb.cpu())
            all_text_embs.append(text_emb.cpu())
            all_texts.extend(texts)
            all_images.extend(images)
            all_img_ids.extend(img_ids)

    all_image_embs = torch.cat(all_image_embs)
    all_text_embs = torch.cat(all_text_embs)
    coarse_sim = all_image_embs @ all_text_embs.T
    return all_texts, all_images, all_img_ids, all_image_embs, all_text_embs, coarse_sim


def rerank_image_to_text_strict(all_texts, all_images, all_image_embs, coarse_sim, alpha=0.9, top_k=20):
    num_items = len(all_texts)
    fused_scores = torch.full((num_items, num_items), -1e9)
    final_scores = torch.full((num_items, num_items), -1e9)

    model.eval()
    with torch.no_grad():
        for i in tqdm(range(num_items), desc="Re-ranking"):
            candidates = coarse_sim[i].topk(top_k).indices.tolist()
            cand_texts = [all_texts[j] for j in candidates]

            image_enc = image_processor(images=[all_images[i]] * len(candidates), return_tensors="pt")
            text_enc = tokenizer(cand_texts, padding="max_length", truncation=True, max_length=cfg.max_text_len, return_tensors="pt")

            _, image_tokens = model.encode_image(image_enc["pixel_values"].to(device))
            _, text_tokens = model.encode_text(text_enc["input_ids"].to(device), text_enc["attention_mask"].to(device))
            cand_fused_emb = model.compute_fused_embedding(text_tokens, image_tokens, text_enc["attention_mask"].to(device))

            query_image_emb = all_image_embs[i].unsqueeze(0)
            scores = (query_image_emb @ cand_fused_emb.cpu().T).squeeze(0)

            for rank, j in enumerate(candidates):
                fused_scores[i, j] = scores[rank]
                final_scores[i, j] = alpha * coarse_sim[i, j] + (1 - alpha) * scores[rank]

    return fused_scores, final_scores


def rerank_text_to_image_strict(all_texts, all_images, all_text_embs, coarse_sim, alpha=0.9, top_k=20):
    num_items = len(all_texts)
    fused_scores = torch.full((num_items, num_items), -1e9)
    final_scores = torch.full((num_items, num_items), -1e9)
    coarse_t = coarse_sim.T

    model.eval()
    with torch.no_grad():
        for i in tqdm(range(num_items), desc="Re-ranking T2I"):
            candidates = coarse_t[i].topk(top_k).indices.tolist()
            cand_images = [all_images[j] for j in candidates]
            text_batch = [all_texts[i]] * len(candidates)

            image_enc = image_processor(images=cand_images, return_tensors="pt")
            text_enc = tokenizer(text_batch, padding="max_length", truncation=True, max_length=cfg.max_text_len, return_tensors="pt")

            _, image_tokens = model.encode_image(image_enc["pixel_values"].to(device))
            _, text_tokens = model.encode_text(text_enc["input_ids"].to(device), text_enc["attention_mask"].to(device))
            cand_fused_emb = model.compute_fused_embedding(text_tokens, image_tokens, text_enc["attention_mask"].to(device))

            query_text_emb = all_text_embs[i].unsqueeze(0)
            scores = (query_text_emb @ cand_fused_emb.cpu().T).squeeze(0)

            for rank, j in enumerate(candidates):
                fused_scores[i, j] = scores[rank]
                final_scores[i, j] = alpha * coarse_t[i, j] + (1 - alpha) * scores[rank]

    return fused_scores, final_scores


all_texts, all_images, all_img_ids, all_image_embs, all_text_embs, coarse_sim = collect_embeddings(test_data, num_samples=cfg.eval_num_samples)
fused_scores_i2t, final_scores_i2t = rerank_image_to_text_strict(all_texts, all_images, all_image_embs, coarse_sim, alpha=cfg.blend_alpha, top_k=cfg.eval_top_k)
fused_scores_t2i, final_scores_t2i = rerank_text_to_image_strict(all_texts, all_images, all_text_embs, coarse_sim, alpha=cfg.blend_alpha, top_k=cfg.eval_top_k)

print(f"\n[Image-to-Text] Coarse Retrieval")
print(f"Recall@1 (1-to-1): {recall_at_k(coarse_sim, 1):.4f}")
print(f"Recall@5 (1-to-1): {recall_at_k(coarse_sim, 5):.4f}")
print(f"Recall@10 (1-to-1): {recall_at_k(coarse_sim, 10):.4f}")
print(f"Recall@1 (group): {recall_at_k(coarse_sim, 1, all_img_ids):.4f}")
print(f"Recall@5 (group): {recall_at_k(coarse_sim, 5, all_img_ids):.4f}")
print(f"Recall@10 (group): {recall_at_k(coarse_sim, 10, all_img_ids):.4f}")

print(f"\n[Image-to-Text] Fused-Embedding Re-rank (strict, alpha=0.0)")
print(f"Recall@1 (1-to-1): {recall_at_k(fused_scores_i2t, 1):.4f}")
print(f"Recall@5 (1-to-1): {recall_at_k(fused_scores_i2t, 5):.4f}")
print(f"Recall@10 (1-to-1): {recall_at_k(fused_scores_i2t, 10):.4f}")
print(f"Recall@1 (group): {recall_at_k(fused_scores_i2t, 1, all_img_ids):.4f}")
print(f"Recall@5 (group): {recall_at_k(fused_scores_i2t, 5, all_img_ids):.4f}")
print(f"Recall@10 (group): {recall_at_k(fused_scores_i2t, 10, all_img_ids):.4f}")

print(f"\n[Image-to-Text] Blended Retrieval (alpha={cfg.blend_alpha:.2f})")
print(f"Recall@1 (1-to-1): {recall_at_k(final_scores_i2t, 1):.4f}")
print(f"Recall@5 (1-to-1): {recall_at_k(final_scores_i2t, 5):.4f}")
print(f"Recall@10 (1-to-1): {recall_at_k(final_scores_i2t, 10):.4f}")
print(f"Recall@1 (group): {recall_at_k(final_scores_i2t, 1, all_img_ids):.4f}")
print(f"Recall@5 (group): {recall_at_k(final_scores_i2t, 5, all_img_ids):.4f}")
print(f"Recall@10 (group): {recall_at_k(final_scores_i2t, 10, all_img_ids):.4f}")

coarse_t2i = coarse_sim.T
print(f"\n[Text-to-Image] Coarse Retrieval")
print(f"Recall@1 (1-to-1): {recall_at_k(coarse_t2i, 1):.4f}")
print(f"Recall@5 (1-to-1): {recall_at_k(coarse_t2i, 5):.4f}")
print(f"Recall@10 (1-to-1): {recall_at_k(coarse_t2i, 10):.4f}")
print(f"Recall@1 (group): {recall_at_k(coarse_t2i, 1, all_img_ids):.4f}")
print(f"Recall@5 (group): {recall_at_k(coarse_t2i, 5, all_img_ids):.4f}")
print(f"Recall@10 (group): {recall_at_k(coarse_t2i, 10, all_img_ids):.4f}")

print(f"\n[Text-to-Image] Fused-Embedding Re-rank (strict, alpha=0.0)")
print(f"Recall@1 (1-to-1): {recall_at_k(fused_scores_t2i, 1):.4f}")
print(f"Recall@5 (1-to-1): {recall_at_k(fused_scores_t2i, 5):.4f}")
print(f"Recall@10 (1-to-1): {recall_at_k(fused_scores_t2i, 10):.4f}")
print(f"Recall@1 (group): {recall_at_k(fused_scores_t2i, 1, all_img_ids):.4f}")
print(f"Recall@5 (group): {recall_at_k(fused_scores_t2i, 5, all_img_ids):.4f}")
print(f"Recall@10 (group): {recall_at_k(fused_scores_t2i, 10, all_img_ids):.4f}")

print(f"\n[Text-to-Image] Blended Retrieval (alpha={cfg.blend_alpha:.2f})")
print(f"Recall@1 (1-to-1): {recall_at_k(final_scores_t2i, 1):.4f}")
print(f"Recall@5 (1-to-1): {recall_at_k(final_scores_t2i, 5):.4f}")
print(f"Recall@10 (1-to-1): {recall_at_k(final_scores_t2i, 10):.4f}")
print(f"Recall@1 (group): {recall_at_k(final_scores_t2i, 1, all_img_ids):.4f}")
print(f"Recall@5 (group): {recall_at_k(final_scores_t2i, 5, all_img_ids):.4f}")
print(f"Recall@10 (group): {recall_at_k(final_scores_t2i, 10, all_img_ids):.4f}")


Re-ranking T2I: 100%|██████████| 1000/1000 [02:14<00:00,  7.43it/s]



[Image-to-Text] Coarse Retrieval
Recall@1 (1-to-1): 0.1430
Recall@5 (1-to-1): 0.6170
Recall@10 (1-to-1): 0.8020
Recall@1 (group): 0.1430
Recall@5 (group): 0.6170
Recall@10 (group): 0.8020

[Image-to-Text] Fused-Embedding Re-rank (strict, alpha=0.0)
Recall@1 (1-to-1): 0.1420
Recall@5 (1-to-1): 0.6100
Recall@10 (1-to-1): 0.7850
Recall@1 (group): 0.1420
Recall@5 (group): 0.6100
Recall@10 (group): 0.7850

[Image-to-Text] Blended Retrieval (alpha=0.50)
Recall@1 (1-to-1): 0.1450
Recall@5 (1-to-1): 0.6220
Recall@10 (1-to-1): 0.7990
Recall@1 (group): 0.1450
Recall@5 (group): 0.6220
Recall@10 (group): 0.7990

[Text-to-Image] Coarse Retrieval
Recall@1 (1-to-1): 0.1370
Recall@5 (1-to-1): 0.6500
Recall@10 (1-to-1): 0.8230
Recall@1 (group): 0.1300
Recall@5 (group): 0.6500
Recall@10 (group): 0.8230

[Text-to-Image] Fused-Embedding Re-rank (strict, alpha=0.0)
Recall@1 (1-to-1): 0.1310
Recall@5 (1-to-1): 0.6510
Recall@10 (1-to-1): 0.8090
Recall@1 (group): 0.1302
Recall@5 (group): 0.6510
Recall@10 (gr